In [2]:
!pip install yfinance
!pip install pandas
from pulp import *
import numpy as np
import yfinance as yf
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
from scipy.optimize import minimize

In [2]:
ticker = "^GSPC"
start_date = "2023-11-01"
sp500_data = yf.download(ticker, start=start_date)
print(sp500_data.head())

/tmp/ipykernel_248/875076730.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500_data = yf.download(ticker, start=start_date)
[*********************100%***********************]  1 of 1 completed

Price             Close         High          Low         Open      Volume
Ticker            ^GSPC        ^GSPC        ^GSPC        ^GSPC       ^GSPC
Date                                                                      
2023-11-01  4237.859863  4245.640137  4197.740234  4201.270020  4224900000
2023-11-02  4317.779785  4319.720215  4268.259766  4268.259766  4669780000
2023-11-03  4358.339844  4373.620117  4334.229980  4334.229980  4570960000
2023-11-06  4365.979980  4372.209961  4347.529785  4364.270020  3656340000
2023-11-07  4378.379883  4386.259766  4355.410156  4366.209961  3791230000


In [3]:
print(f"Type of sp500_data: {type(sp500_data)}")
display(sp500_data.head())

Type of sp500_data: <class 'pandas.core.frame.DataFrame'>


Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
2023-11-01,4237.859863,4245.640137,4197.740234,4201.270020,4224900000
2023-11-02,4317.779785,4319.720215,4268.259766,4268.259766,4669780000
2023-11-03,4358.339844,4373.620117,4334.229980,4334.229980,4570960000
2023-11-06,4365.979980,4372.209961,4347.529785,4364.270020,3656340000
2023-11-07,4378.379883,4386.259766,4355.410156,4366.209961,3791230000


In [5]:
!pip install BeautifulSoup

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [6]:
import requests
from bs4 import BeautifulSoup
import csv
from datetime import datetime

BASE_URL = "https://rollcall.com/factbase/trump/topic/social/?platform=all&sort=date&sort_order=desc&page={}"
CUTOFF = datetime(2023, 11, 1)

def parse_date(date_str):
    # Example format: "January 15, 2024"
    return datetime.strptime(date_str.strip(), "%B %d, %Y")

def scrape_page(page_num):
    url = BASE_URL.format(page_num)
    r = requests.get(url)
    soup = BeautifulSoup(r.text, "html.parser")

    posts = soup.select("div.factbase-post")  # container for each post
    results = []

    for post in posts:
        date_tag = post.select_one(".factbase-post-date")
        text_tag = post.select_one(".factbase-post-text")
        platform_tag = post.select_one(".factbase-post-platform")
        link_tag = post.select_one("a")

        if not date_tag or not text_tag:
            continue

        date = parse_date(date_tag.text)
        if date < CUTOFF:
            return results, True  # hit cutoff → stop scraping

        text = text_tag.text.strip()
        platform = platform_tag.text.strip() if platform_tag else "Unknown"
        link = link_tag["href"] if link_tag else ""

        results.append({
            "date": date.strftime("%Y-%m-%d"),
            "platform": platform,
            "text": text,
            "link": link
        })

    return results, False

def scrape_all():
    all_posts = []
    page = 1
    stop = False

    while not stop:
        print(f"Scraping page {page}...")
        posts, stop = scrape_page(page)
        all_posts.extend(posts)
        page += 1

    return all_posts

def save_csv(posts, filename="trump_posts.csv"):
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["date", "platform", "text", "link"])
        writer.writeheader()
        writer.writerows(posts)

if __name__ == "__main__":
    posts = scrape_all()
    save_csv(posts)
    print(f"Saved {len(posts)} posts to trump_posts.csv")

Streaming output truncated to the last 5000 lines.
Scraping page 1702...
Scraping page 1703...
Scraping page 1704...
Scraping page 1705...
Scraping page 1706...
Scraping page 1707...
Scraping page 1708...
Scraping page 1709...
Scraping page 1710...
Scraping page 1711...
Scraping page 1712...
Scraping page 1713...
Scraping page 1714...
Scraping page 1715...
Scraping page 1716...
Scraping page 1717...
Scraping page 1718...
Scraping page 1719...
Scraping page 1720...
Scraping page 1721...
Scraping page 1722...
Scraping page 1723...
Scraping page 1724...
Scraping page 1725...
Scraping page 1726...
Scraping page 1727...
Scraping page 1728...
Scraping page 1729...
Scraping page 1730...
Scraping page 1731...
Scraping page 1732...
Scraping page 1733...
Scraping page 1734...
Scraping page 1735...
Scraping page 1736...
Scraping page 1737...
Scraping page 1738...
Scraping page 1739...
Scraping page 1740...
Scraping page 1741...
Scraping page 1742...
Scraping page 1743...
Scraping page 1744...
Scr

KeyboardInterrupt: 

In [4]:
import pandas as pd
try:
    # Assuming the uploaded TSV file is named 'uploaded_data.tsv'
    # The first column is indexed as 0
    tsv_df = pd.read_csv('uploaded_data.tsv', sep='\t', dtype={0: str})
    print("TSV file loaded successfully into a DataFrame.")
    print(f"DataFrame shape: {tsv_df.shape}")
    print("First 5 rows of the DataFrame:")
    display(tsv_df.head())
    print("Data types of the DataFrame columns:")
    print(tsv_df.dtypes)
except FileNotFoundError:
    print("Error: 'uploaded_data.tsv' not found. Please make sure you have uploaded the TSV file and its name is correct.")
except Exception as e:
    print(f"An error occurred: {e}")

An error occurred: 'utf-8' codec can't decode byte 0x85 in position 37: invalid start byte
